# NIH Tracker pipeline notebook (v19.2.1; extends v19.2)

This notebook performs one standard tracker refresh. It contains no review workbench and no destructive maintenance controls. Compile, input, lock, and card-parse checks always run. External API/service smoke checks and live evidence refreshes run only when the explicit external-network option is selected. A scope-aware crosswalk preflight runs before normal card mutations, and idempotent transformation steps treat zero required changes as success. Every subprocess streams to the notebook and is also retained under `runs/notebook_debug/pipeline/`.

The tracker now has one operational environment: **production**. The final cell creates or applies a provenance-bound GitHub publication plan.


## 1. Start the production runtime

Run this cell first. The project root is fixed at `/content/drive/MyDrive/NIH_Tracker`; there is no environment selector. Production card writes remain disabled unless the acknowledgement box is checked.


In [ ]:
NOTEBOOK_NAME = "pipeline"

ENVIRONMENT = "prod"
ENABLE_PROD_WRITES = False  # @param {type:"boolean"}

import os
import sys
from pathlib import Path

ROOTS = {"prod": "/content/drive/MyDrive/NIH_Tracker"}
os.environ["NIH_TRACKER_PRODUCTION_ONLY"] = "1"
os.environ["NIH_TRACKER_ROOT_PROD"] = ROOTS["prod"]
os.environ["NIH_TRACKER_ENV"] = "prod"
os.environ["NIH_TRACKER_SHARED_CACHE_ROOT"] = "/content/drive/MyDrive/NIH_Tracker_shared/cache"
os.environ.pop("NIH_TRACKER", None)
os.environ.pop("NIH_TRACKER_ALLOW_PROJECT_OVERRIDE", None)
os.environ.pop("NIH_TRACKER_ROOT_DEV", None)
os.environ.pop("NIH_TRACKER_ROOT_STAGING", None)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

SCRIPTS = Path(ROOTS["prod"]) / "scripts"
if not SCRIPTS.is_dir():
    raise FileNotFoundError(
        f"Missing {SCRIPTS}. Run the maintenance notebook's production release installer first."
    )
previous_scripts = globals().get("_NIH_TRACKER_PREVIOUS_SCRIPTS")
if previous_scripts and previous_scripts in sys.path:
    sys.path.remove(previous_scripts)
if str(SCRIPTS) in sys.path:
    sys.path.remove(str(SCRIPTS))
sys.path.insert(0, str(SCRIPTS))
_NIH_TRACKER_PREVIOUS_SCRIPTS = str(SCRIPTS)
for module_name in list(sys.modules):
    if (
        module_name in {"tracker_gcp_backend", "evidence_store", "evidence_collector", "capability_assessment", "gcp_review", "gcp_workbench", "gcp_config", "tracker_cache_bridge", "gcp_review_queue", "tracker_claude_backend", "maintenance_widgets", "run_provenance", "code_provenance", "step_project_inventory", "common", "identity_hygiene", "interoperability_schema", "resource_profile_schema", "policy_context_schema", "repository_status_schema", "publisher_context_schema", "notebook_runtime", "review_widgets", "review_workflow", "automated_review", "verify_external_ids", "catalog_schema", "catalog_vocab", "catalog_claims", "catalog_graph", "catalog_resolver", "catalog_history", "catalog_exports"}
        or module_name.startswith("step_")
    ):
        sys.modules.pop(module_name, None)

from notebook_runtime import NotebookRuntime
RUNTIME = NotebookRuntime.create(
    notebook_name=NOTEBOOK_NAME,
    env="prod",
    enable_prod_writes=ENABLE_PROD_WRITES,
    roots=ROOTS,
)
common = RUNTIME.common
print(f"Ready: notebook={NOTEBOOK_NAME} | production | project={common.PROJECT}")
print(f"Debug session: {RUNTIME.debug_dir}")


## 2. Verify runtime state

This cell records local runtime diagnostics only. API credentials are not loaded here. They are loaded just-in-time only when **Refresh external API/web evidence** is selected in the pipeline cell.


In [ ]:
SECRET_STATUS = {}
DIAGNOSTICS = RUNTIME.show_diagnostics()
print("External API credentials have not been loaded. The default pipeline is offline/cached-only.")


## 3. Run the standard pipeline

Choose **Validate/propose (no card writes)** to produce run artifacts, proposals, diagnostics, and a rebuilt preview without changing cards. Choose **Apply approved changes** to permit each step's guarded `--apply` path. The sequence is intentionally fixed; review decisions are handled in the review notebook.

Checking `RUN_PIPELINE` always runs the local smoke suite first. The external-service smoke test is skipped unless the external-evidence checkbox is selected. A scope-aware crosswalk referential-integrity preflight then runs before the first normal card-mutating step. Rows without `registry_name` are reported as non-applicable because this join does not consume them. An unreviewed stale target remains blocking; a receipted `DEFER` is withheld from the join and carried as a nonblocking advisory. Structural conflicts and malformed in-scope rows still block. No candidate is accepted automatically.

The fixed sequence includes interoperability, resource-profile, repository-status, publisher-policy, and policy-context builds, followed by a deterministic knowledge-graph compiler and task-oriented public site build. The interoperability step normalizes API evidence, endpoint and documentation links, standards, PID systems, and resource-specific participant GUID systems. The resource-profile step uses exact linked FAIRsharing/re3data records plus structured homepage metadata to add descriptions, resource and data-asset types, subject areas, formats, and organizational stewardship evidence. The repository-status step separates operational availability from broad, limited, paused, or closed submission intake using BMIC plus reviewed resource-specific overrides. The publisher-policy step records mandatory, recommended, preferred, fallback, example, and directory-referral relationships without collapsing them into one endorsement claim. The policy-context step uses pinned NIH CADR and dbGaP study snapshots and keeps official CADR listing, dbGaP External Data Source evidence, and public-study text associations analytically separate. It does not infer physical data residency or EDS status from study text. The v19 catalog emits typed nodes, relationships, evidence-bearing claims, controlled vocabularies, change history, and static JSON/JSONL/CSV/JSON-LD exports. Public completeness or maturity scores are not produced. Expected gate failures stop cleanly with a persisted status record instead of a notebook traceback. Valid no-op transforms pass.


In [ ]:
RUN_PIPELINE = False  # @param {type:"boolean"}
PIPELINE_MODE = "Validate/propose (no card writes)"  # @param ["Validate/propose (no card writes)", "Apply approved changes"]
REFRESH_EXTERNAL_EVIDENCE = False  # @param {type:"boolean"}
# False (default): no SciCrunch, FAIRsharing, DataCite, re3data, or repository-homepage requests.
# True: run live service smoke checks, identifier verification/search, and cache refreshes.

from pathlib import Path
import datetime as dt
import traceback
import pandas as pd
from IPython.display import display

APPLY_CHANGES = PIPELINE_MODE == "Apply approved changes"
NETWORK_MODE = "live_external_refresh" if REFRESH_EXTERNAL_EVIDENCE else "offline_cached_only"
os.environ["NIH_TRACKER_EXTERNAL_NETWORK_ENABLED"] = "1" if REFRESH_EXTERNAL_EVIDENCE else "0"
PIPELINE_SUCCESS_PATH = RUNTIME.debug_dir / "pipeline_success.json"
PIPELINE_STATUS_PATH = RUNTIME.debug_dir / "pipeline_status.json"
PIPELINE_EXCEPTION_PATH = RUNTIME.debug_dir / "pipeline_exception.txt"
REQUIRED_INPUTS = [
    "BMIC_download.csv",
    "Tools467results.csv",
    "crosswalk.csv",
    "final_ultimate_enhanced_repos.csv",
    "repo_api_results_v4_230_strict_endpoints.csv",
    "FAIRSHARING_EXPORT.csv",
]

def _build_pipeline_steps():
    rrid_args = ["--tools", "Tools467results.csv"]
    fairsharing_args = ["--export", "FAIRSHARING_EXPORT.csv"]
    interoperability_args = ["--probe", "repo_api_results_v4_230_strict_endpoints.csv"]
    resource_profile_args = ["--fairsharing-export", "FAIRSHARING_EXPORT.csv"]
    if REFRESH_EXTERNAL_EVIDENCE:
        rrid_args.append("--verify")
        fairsharing_args.append("--verify")
        interoperability_args.extend(["--re3data-policy", "missing"])
        resource_profile_args.extend(
            ["--fairsharing-policy", "missing", "--website-policy", "missing"]
        )
    else:
        interoperability_args.extend(["--re3data-policy", "none"])
        resource_profile_args.extend(
            ["--fairsharing-policy", "none", "--website-policy", "none"]
        )

    steps = [
        (
            "crosswalk_preflight",
            "step_build_links.py",
            ["--check-crosswalk-only"],
            False,
        ),
        ("build_names", "step_build_names.py", [], True),
        ("clean_aliases", "step_clean_aliases.py", [], True),
        (
            "build_links",
            "step_build_links.py",
            ["--registry", "final_ultimate_enhanced_repos.csv"],
            True,
        ),
        ("build_rrids", "step_build_rrids.py", rrid_args, True),
        ("build_fairsharing", "step_build_fairsharing.py", fairsharing_args, True),
        (
            "reconcile_ids",
            "step_reconcile_ids.py",
            ["--probe", "repo_api_results_v4_230_strict_endpoints.csv"],
            True,
        ),
    ]
    if REFRESH_EXTERNAL_EVIDENCE:
        steps.append(
            (
                "verify_external_ids",
                "verify_external_ids.py",
                ["--mode", "search", "--only-missing", "--verbose"],
                True,
            )
        )
    steps.extend(
        [
            (
                "build_interoperability",
                "step_build_interoperability.py",
                interoperability_args,
                True,
            ),
            (
                "build_resource_profiles",
                "step_build_resource_profiles.py",
                resource_profile_args,
                True,
            ),
            (
                "build_repository_status",
                "step_build_repository_status.py",
                ["--bmic", "BMIC_download.csv"],
                True,
            ),
            ("build_policy_context", "step_build_policy_context.py", [], True),
            ("build_publisher_context", "step_build_publisher_context.py", [], True),
            ("bmic_diff", "step_bmic_diff.py", ["--bmic", "BMIC_download.csv"], False),
            (
                "refresh_plan",
                "step_refresh_plan.py",
                ["--blocks", "identity,bmic,repository_status,resource_profile,interoperability,policy_context,publisher_context,pid_culture", "--stale-months", "12"],
                False,
            ),
            (
                "build_spine",
                "step_build_spine.py",
                ["--blocks", "identity,bmic,repository_status,resource_profile,interoperability,policy_context,publisher_context,pid_culture"],
                False,
            ),
            ("build_catalog", "step_build_catalog.py", [], False),
            ("build_frontend", "step_build_frontend.py", [], False),
        ]
    )
    return steps


PIPELINE_STEPS = _build_pipeline_steps()


def _pipeline_status(ok, stage, message, results=None, **extra):
    record = {
        "schema": "nih-tracker-pipeline-status/v1",
        "session_id": RUNTIME.session_id,
        "environment": common.ENV,
        "project": str(Path(common.PROJECT).resolve()),
        "pipeline_mode": PIPELINE_MODE,
        "network_mode": NETWORK_MODE,
        "external_network_enabled": bool(REFRESH_EXTERNAL_EVIDENCE),
        "ok": bool(ok),
        "stage": str(stage),
        "message": str(message),
        "recorded_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "results": list(results or []),
        **extra,
    }
    common.atomic_write_json(PIPELINE_STATUS_PATH, record)
    banner = "PIPELINE COMPLETE" if ok else "PIPELINE STOPPED"
    print("\n" + "=" * 100)
    print(f"{banner} | stage={stage} | env={common.ENV}")
    print(message)
    print(f"Status record: {PIPELINE_STATUS_PATH}")
    if not ok:
        print("No release-eligibility marker was created.")
    print("=" * 100)
    return record


def _execute_pipeline():
    if not RUN_PIPELINE:
        print("Pipeline not started. Check RUN_PIPELINE when the selected environment is ready.")
        return {
            "ok": False,
            "stage": "not_started",
            "message": "RUN_PIPELINE is not checked.",
            "results": [],
        }

    if PIPELINE_SUCCESS_PATH.exists():
        PIPELINE_SUCCESS_PATH.unlink()
    if PIPELINE_EXCEPTION_PATH.exists():
        PIPELINE_EXCEPTION_PATH.unlink()

    results = []
    try:
        if APPLY_CHANGES and common.ENV == "prod" and not ENABLE_PROD_WRITES:
            return _pipeline_status(
                False,
                "configuration",
                "Apply mode in production requires ENABLE_PROD_WRITES in the setup cell.",
                results,
            )

        if REFRESH_EXTERNAL_EVIDENCE:
            RUNTIME.load_pipeline_secrets()
            print("External evidence refresh: ENABLED. Live API/web requests are permitted for this run.")
        else:
            RUNTIME.clear_pipeline_secrets()
            print("External evidence refresh: DISABLED. This run uses local inputs and existing caches only.")
        smoke = RUNTIME.smoke_suite(
            required_inputs=REQUIRED_INPUTS,
            required_project_files=[
                "frontend.html",
                "reference/nih_cadr_registry.csv",
                "reference/dbGaP_Studies.csv",
                "reference/dbgap_eds_definition.json",
                "reference/re3data_confirmed_mappings.csv",
                "reference/publisher_policy_sources.csv",
                "reference/publisher_repository_mentions.csv",
                "reference/repository_status_overrides.csv",
                "reference/catalog_vocabularies_v1.json",
                "schemas/catalog_node.schema.json",
                "schemas/catalog_edge.schema.json",
                "schemas/catalog_claim.schema.json",
                "schemas/data_asset.schema.json",
            ],
            external=REFRESH_EXTERNAL_EVIDENCE,
        )
        if not smoke["passed"]:
            return _pipeline_status(
                False,
                "smoke_suite",
                f"Mandatory smoke suite failed; inspect {RUNTIME.debug_dir / 'smoke_report.json'}.",
                results,
                smoke_report=str(RUNTIME.debug_dir / "smoke_report.json"),
            )

        if not common.all_cards():
            if not APPLY_CHANGES:
                return _pipeline_status(
                    False,
                    "materialize_cards",
                    "No cards are present. Seed the environment in the maintenance notebook "
                    "or run this pipeline in apply mode for initial materialization.",
                    results,
                )
            materialize = RUNTIME.run_script(
                "step_materialize_cards.py",
                ["--bmic", "BMIC_download.csv", "--attach", "interoperability"],
                label="materialize_cards",
                writes_cards=True,
            )
            results.append(materialize.to_dict())
            if not materialize.ok:
                return _pipeline_status(
                    False,
                    "materialize_cards",
                    f"Materialization failed; inspect {materialize.log_path}",
                    results,
                    failing_log=str(materialize.log_path),
                )

        for label, script, base_arguments, may_write_cards in PIPELINE_STEPS:
            arguments = list(base_arguments)
            writes_cards = bool(APPLY_CHANGES and may_write_cards)
            if writes_cards:
                arguments.append("--apply")
            result = RUNTIME.run_script(
                script,
                arguments,
                label=label,
                writes_cards=writes_cards,
                load_script_secrets=REFRESH_EXTERNAL_EVIDENCE,
            )
            results.append(result.to_dict())
            if not result.ok:
                guidance = (
                    "Open the latest crosswalk_integrity.csv in the review workbench, "
                    "record and apply evidence-based replacements/remaps or reviewed DEFER "
                    "dispositions, then rerun the full pipeline."
                    if label == "crosswalk_preflight"
                    else "Inspect the subprocess log and its run artifacts before rerunning."
                )
                return _pipeline_status(
                    False,
                    label,
                    f"Step failed or its sanity gate did not pass. {guidance}",
                    results,
                    failing_log=str(result.log_path),
                    failing_result=str(result.result_path),
                )

        build_path = Path(common.BUILD) / "index.html"
        build_sha256 = common.sha256_file(build_path)
        if not build_sha256:
            return _pipeline_status(
                False,
                "release_artifact",
                f"Pipeline steps passed but the release artifact is missing or unreadable: {build_path}",
                results,
            )

        success_record = {
            "schema": "nih-tracker-pipeline-success/v1",
            "session_id": RUNTIME.session_id,
            "environment": common.ENV,
            "project": str(Path(common.PROJECT).resolve()),
            "pipeline_mode": PIPELINE_MODE,
            "network_mode": NETWORK_MODE,
            "external_network_enabled": bool(REFRESH_EXTERNAL_EVIDENCE),
            "completed_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "build_path": str(build_path.resolve()),
            "build_sha256": build_sha256,
            "steps": [row["label"] for row in results],
            "results": results,
        }
        common.atomic_write_json(PIPELINE_SUCCESS_PATH, success_record)
        status = _pipeline_status(
            True,
            "complete",
            "All subprocess and ledger gates passed.",
            results,
            release_eligibility_marker=str(PIPELINE_SUCCESS_PATH),
            build_path=str(build_path.resolve()),
            build_sha256=build_sha256,
        )
        print(f"Release eligibility marker: {PIPELINE_SUCCESS_PATH}")
        print(f"Built index SHA-256: {build_sha256}")
        return status
    except Exception as exc:
        common.atomic_write_text(PIPELINE_EXCEPTION_PATH, traceback.format_exc())
        return _pipeline_status(
            False,
            "unexpected_exception",
            f"{type(exc).__name__}: {exc}",
            results,
            traceback_file=str(PIPELINE_EXCEPTION_PATH),
        )


PIPELINE_STATUS = _execute_pipeline()
if PIPELINE_STATUS.get("results"):
    result_frame = pd.DataFrame(PIPELINE_STATUS["results"])
    columns = [
        column
        for column in [
            "label",
            "ok",
            "returncode",
            "gate_passed",
            "duration_seconds",
            "log_path",
        ]
        if column in result_frame.columns
    ]
    display(result_frame[columns])
RUNTIME.write_session_summary()


## 4. Inspect the evidence produced by this session

This cell shows recent ledger rows, unresolved issues ordered by severity and recurrence, and the persisted debug result records. It does not rerun any pipeline step.


In [ ]:
import pandas as pd
from IPython.display import display

recent_ledger = RUNTIME.recent_ledger(20)
open_issues = RUNTIME.open_issues(250)
run_results = RUNTIME.run_results()

print(f"Debug directory: {RUNTIME.debug_dir}")
print(f"Recent ledger rows: {len(recent_ledger)}")
if recent_ledger:
    display(pd.DataFrame(recent_ledger))
print(f"Open issues: {len(open_issues)}")
if open_issues:
    display(pd.DataFrame(open_issues))
print(f"Subprocesses in this notebook session: {len(run_results)}")
if run_results:
    display(pd.DataFrame(run_results))
RUNTIME.write_session_summary()


## 5. Publish production

This is the final operational cell. **Create a sealed release plan** is enabled only after the fixed production pipeline completes successfully in the current session and captures hash-pinned bytes plus pipeline provenance for review. If the session is not release-eligible, the cell prints the blocking pipeline stage and exits without a traceback. **Apply the newest unapplied plan after review** selects only a plan carrying verified pipeline provenance; when none exists, it writes `release_status.json`, explains that a plan must be created first, and exits without changing GitHub.

Production GitHub planning reads `GITHUB_REPOSITORY`; publication also requires `GITHUB_TOKEN`. Set `GITHUB_PUBLISH_DIR` to `docs`; GitHub Pages must be configured for the selected branch and `/docs`. Optional identity settings are `GITHUB_COMMIT_NAME` and `GITHUB_COMMIT_EMAIL`.


In [ ]:
RELEASE_ACTION = "Create a sealed release plan"  # @param ["Create a sealed release plan", "Apply the newest unapplied plan after review"]

import datetime as dt
import hashlib
import importlib
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

APPLY_RELEASE = RELEASE_ACTION == "Apply the newest unapplied plan after review"
PIPELINE_SUCCESS_PATH = RUNTIME.debug_dir / "pipeline_success.json"
RELEASE_STATUS_PATH = RUNTIME.debug_dir / "release_status.json"
RELEASE_PROVENANCE_NAME = "pipeline_release_provenance.json"
RELEASE_PROVENANCE_SEAL = "pipeline_release_provenance.sha256"


def _canonical_sha256(value):
    encoded = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def _load_current_pipeline_success():
    if not PIPELINE_SUCCESS_PATH.is_file():
        raise RuntimeError(
            "Release planning is disabled because this notebook session has no successful "
            "end-to-end pipeline marker. Run the standard pipeline to completion first."
        )
    success = json.loads(PIPELINE_SUCCESS_PATH.read_text(encoding="utf-8"))
    if success.get("schema") != "nih-tracker-pipeline-success/v1":
        raise RuntimeError("Unsupported pipeline success marker schema.")
    if success.get("session_id") != RUNTIME.session_id:
        raise RuntimeError("Pipeline success marker belongs to a different notebook session.")
    if success.get("environment") != common.ENV:
        raise RuntimeError("Pipeline success marker belongs to a different environment.")
    if Path(success.get("project", "")).resolve() != Path(common.PROJECT).resolve():
        raise RuntimeError("Pipeline success marker belongs to a different project root.")
    expected_steps = [label for label, _, _, _ in PIPELINE_STEPS]
    observed_steps = success.get("steps")
    permitted_sequences = [expected_steps, ["materialize_cards", *expected_steps]]
    if observed_steps not in permitted_sequences:
        raise RuntimeError(
            "Pipeline success marker does not contain the complete fixed pipeline sequence."
        )
    build_path = Path(success.get("build_path", "")).resolve()
    expected_build = (Path(common.BUILD) / "index.html").resolve()
    if build_path != expected_build:
        raise RuntimeError("Pipeline success marker points to an unexpected build artifact.")
    current_hash = common.sha256_file(build_path)
    if not current_hash or current_hash != success.get("build_sha256"):
        raise RuntimeError(
            "build/index.html changed after the successful pipeline run. Rerun the full "
            "pipeline before creating a release plan."
        )
    return success


def _seal_release_provenance(plan_path, success):
    plan_path = Path(plan_path).resolve()
    record = {
        "schema": "nih-tracker-pipeline-release-provenance/v1",
        "environment": common.ENV,
        "project": str(Path(common.PROJECT).resolve()),
        "release_plan": str(plan_path),
        "release_plan_sha256": common.sha256_file(plan_path),
        "pipeline_success": success,
        "pipeline_success_sha256": _canonical_sha256(success),
    }
    provenance_path = plan_path.parent / RELEASE_PROVENANCE_NAME
    common.atomic_write_json(provenance_path, record)
    seal_path = plan_path.parent / RELEASE_PROVENANCE_SEAL
    common.atomic_write_text(seal_path, common.sha256_file(provenance_path) + "\n")
    print(f"Release provenance: {provenance_path}")
    print(f"Release provenance seal: {seal_path}")


def _verify_release_provenance(plan_path):
    plan_path = Path(plan_path).resolve()
    provenance_path = plan_path.parent / RELEASE_PROVENANCE_NAME
    seal_path = plan_path.parent / RELEASE_PROVENANCE_SEAL
    if not provenance_path.is_file() or not seal_path.is_file():
        return False, "missing pipeline release provenance or provenance seal"
    try:
        expected_provenance_hash = seal_path.read_text(encoding="utf-8").strip()
        if expected_provenance_hash != common.sha256_file(provenance_path):
            return False, "pipeline release provenance seal mismatch"
        record = json.loads(provenance_path.read_text(encoding="utf-8"))
        success = record.get("pipeline_success", {})
        plan = json.loads(plan_path.read_text(encoding="utf-8"))
        expected_steps = [label for label, _, _, _ in PIPELINE_STEPS]
        observed_steps = success.get("steps")
        permitted_sequences = [expected_steps, ["materialize_cards", *expected_steps]]
        result_rows = success.get("results", [])
        result_labels = [row.get("label") for row in result_rows]
        result_ok = bool(result_rows) and all(row.get("ok") is True for row in result_rows)
        checks = [
            record.get("schema") == "nih-tracker-pipeline-release-provenance/v1",
            record.get("environment") == common.ENV,
            Path(record.get("project", "")).resolve() == Path(common.PROJECT).resolve(),
            Path(record.get("release_plan", "")).resolve() == plan_path,
            record.get("release_plan_sha256") == common.sha256_file(plan_path),
            record.get("pipeline_success_sha256") == _canonical_sha256(success),
            success.get("schema") == "nih-tracker-pipeline-success/v1",
            success.get("environment") == common.ENV,
            Path(success.get("project", "")).resolve() == Path(common.PROJECT).resolve(),
            observed_steps in permitted_sequences,
            result_labels == observed_steps,
            result_ok,
        ]
        if plan.get("schema") == "nih-tracker-github-publication/v1":
            checks.extend(
                [
                    Path(plan.get("source", "")).resolve()
                    == Path(success.get("build_path", "")).resolve(),
                    plan.get("source_sha256") == success.get("build_sha256"),
                ]
            )
        if not all(checks):
            return False, "pipeline release provenance failed hash, scope, sequence, or build validation"
    except Exception as exc:
        return False, f"invalid pipeline release provenance: {exc}"
    return True, ""


def _find_newest_eligible(candidates, applied_marker):
    rejected = []
    for candidate in candidates:
        applied_path = candidate.parent / applied_marker
        if applied_path.exists():
            rejected.append(
                {
                    "plan": str(candidate),
                    "reason": f"already applied ({applied_path.name})",
                }
            )
            continue
        eligible, reason = _verify_release_provenance(candidate)
        if eligible:
            return candidate, rejected
        rejected.append({"plan": str(candidate), "reason": reason})
    return None, rejected


def _write_release_status(state, message, *, details=None, rejected=None, plan_path=None):
    record = {
        "schema": "nih-tracker-notebook-release-status/v1",
        "recorded_at": dt.datetime.now(dt.timezone.utc).isoformat(),
        "session_id": RUNTIME.session_id,
        "environment": common.ENV,
        "project": str(Path(common.PROJECT).resolve()),
        "requested_action": RELEASE_ACTION,
        "state": state,
        "message": message,
        "plan_path": str(Path(plan_path).resolve()) if plan_path else "",
        "details": details or {},
        "rejected_candidates": rejected or [],
    }
    common.atomic_write_json(RELEASE_STATUS_PATH, record)
    return RELEASE_STATUS_PATH


def _print_release_block(heading, message, *, next_steps, details=None, rejected=None):
    status_path = _write_release_status(
        heading.lower().replace(" ", "_"),
        message,
        details=details,
        rejected=rejected,
    )
    print("\n" + "=" * 100)
    print(heading)
    print(message)
    if details:
        for key, value in details.items():
            if value not in (None, "", [], {}):
                print(f"{key}: {value}")
    if rejected:
        print("Rejected or unavailable plans:")
        for item in rejected[:8]:
            print(f"  - {item['plan']}: {item['reason']}")
        if len(rejected) > 8:
            print(f"  ... {len(rejected) - 8} additional candidate(s) recorded in {status_path}")
    print("Next steps:")
    for index, step in enumerate(next_steps, start=1):
        print(f"  {index}. {step}")
    print(f"Status record: {status_path}")
    print("No repository or environment was changed.")
    print("=" * 100)


pipeline_success = None
RELEASE_ELIGIBLE = True
if not APPLY_RELEASE:
    try:
        pipeline_success = _load_current_pipeline_success()
    except Exception as exc:
        RELEASE_ELIGIBLE = False
        status_path = RUNTIME.debug_dir / "pipeline_status.json"
        pipeline_status = {}
        if status_path.is_file():
            try:
                pipeline_status = json.loads(status_path.read_text(encoding="utf-8"))
            except Exception:
                pipeline_status = {}
        details = {
            "pipeline_stage": pipeline_status.get("stage", "") if pipeline_status else "",
            "pipeline_status": pipeline_status.get("message", "") if pipeline_status else "",
            "pipeline_status_record": str(status_path) if status_path.is_file() else "",
        }
        _print_release_block(
            "RELEASE NOT ELIGIBLE",
            str(exc),
            details=details,
            next_steps=[
                "Run the standard pipeline to completion in this notebook session.",
                "Confirm that pipeline_success.json was created and build/index.html is unchanged.",
                "Run this cell with 'Create a sealed release plan'.",
            ],
        )
        print("No promotion or GitHub publication plan was created.")

if RELEASE_ELIGIBLE:
    if not APPLY_RELEASE:
        RUNTIME.load_secrets(
            [
                "GITHUB_REPOSITORY",
                "GITHUB_BRANCH",
                "GITHUB_PUBLISH_DIR",
                "GITHUB_COMMIT_NAME",
                "GITHUB_COMMIT_EMAIL",
            ],
            required=["GITHUB_REPOSITORY"],
            quiet=False,
        )
        before = set(
            (Path(common.RUNS) / "publish_github").glob("*/publication_plan.json")
        )
        result = RUNTIME.run_script(
            "step_publish_github.py",
            [],
            label="publish_production_to_github",
            required_secrets=["GITHUB_REPOSITORY"],
        )
        if not result.ok:
            raise RuntimeError(f"GitHub publication planning failed; inspect {result.log_path}")
        after = set(
            (Path(common.RUNS) / "publish_github").glob("*/publication_plan.json")
        )
        created = sorted(
            after.difference(before),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if len(created) != 1:
            raise RuntimeError(
                f"Expected exactly one new publication plan; found {len(created)}."
            )
        plan_path = created[0]
        _seal_release_provenance(plan_path, pipeline_success)
        print("Publication plan created. Review it before applying.")
        _write_release_status(
            "plan_created",
            "Created sealed GitHub publication plan.",
            plan_path=plan_path,
        )
    else:
        plans = sorted(
            (Path(common.RUNS) / "publish_github").glob("*/publication_plan.json"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        plan_path, rejected = _find_newest_eligible(plans, "publication_receipt.json")
        if plan_path is None:
            _print_release_block(
                "NO ELIGIBLE RELEASE PLAN",
                "No unapplied GitHub publication plan with verified end-to-end pipeline provenance was found. "
                "The older plans listed below predate the provenance control and cannot be applied.",
                rejected=rejected,
                next_steps=[
                    "Run the fixed production pipeline to completion if the current build is not already release-eligible.",
                    "Set RELEASE_ACTION to 'Create a sealed release plan' and run this cell once.",
                    "Review publication_plan.json, pipeline_release_provenance.json, and both SHA-256 seals.",
                    "Set RELEASE_ACTION to 'Apply the newest unapplied plan after review' and rerun.",
                ],
            )
        else:
            RUNTIME.load_secrets(
                [
                    "GITHUB_TOKEN",
                    "GITHUB_REPOSITORY",
                    "GITHUB_BRANCH",
                    "GITHUB_PUBLISH_DIR",
                    "GITHUB_COMMIT_NAME",
                    "GITHUB_COMMIT_EMAIL",
                ],
                required=["GITHUB_TOKEN"],
                quiet=False,
            )
            print(f"Selected verified publication plan: {plan_path}")
            result = RUNTIME.run_script(
                "step_publish_github.py",
                [
                    "--plan",
                    str(plan_path),
                    "--apply",
                    "--confirm",
                    "PUBLISH_PROD_TO_GITHUB",
                ],
                label="publish_production_to_github",
                required_secrets=["GITHUB_TOKEN"],
            )
            if not result.ok:
                raise RuntimeError(f"GitHub publication failed; inspect {result.log_path}")
            _write_release_status(
                "publication_applied",
                "Applied verified GitHub publication plan.",
                plan_path=plan_path,
                details={"debug_log": str(result.log_path)},
            )

RUNTIME.write_session_summary()


## v19.1 GCP extension notes

Install this v19.1.1 package using the existing production release installer.
The new `requirements-gcp.txt`, source-cache modules, assessment rubric and review
panel are release-managed; existing cards, inputs, snapshots, caches and reviews
are preserved. Use the review workbench's optional GCP panel for execution.
The catalog build remains offline. No cloud resource or recurring job is created
by this notebook automatically. See `GCP_INTEGRATION_AND_ASSESSMENT.md`.
